In [1]:
import numpy as np
import struct
from array import array
from os.path  import join
from time import perf_counter
import tqdm

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [2]:
# local to this repo

from data import build_dataloaders
from model import ConvModel
from util import test_loss, test_accuracy
from optim import GradSignOptimizer    # my custom optimizer

In [3]:
cross_entropy = nn.CrossEntropyLoss()

In [4]:
# Just find out how many params
# We will re-initialize the model in-loop for reproducibility

model = ConvModel()
sum([p.numel() for p in model.parameters()])

66954

In [5]:
# Constant across experimental runs
n_batches = 1000


# GradSign, my custom optimizer

In [6]:
#training loop

# a good range for learning rate with GradSign or Adam
for lr in [0.0001, 0.0003, 0.001, 0.003]:
    # Best: Learning rate: 0.0003; test accuracy: 0.9829

    # Each learning rate trial takes ~2 min at n_batches = 1000
    # so let's show that something is happening while we wait
    pbar = tqdm.tqdm(total=n_batches)
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = GradSignOptimizer(model.named_parameters(), lr=lr)
    model.train()

    # For reproducibility (because optim.__init__() called the torch RNG)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)

            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            pbar.update(1)
            if i == n_batches:
                break

    pbar.close()
    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:47<00:00,  9.32it/s]


Learning rate: 0.0001; test accuracy: 0.9765


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:44<00:00,  9.56it/s]


Learning rate: 0.0003; test accuracy: 0.9829


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:41<00:00,  9.85it/s]


Learning rate: 0.001; test accuracy: 0.9829


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:43<00:00,  9.69it/s]


Learning rate: 0.003; test accuracy: 0.977


# Stochastic gradient descent

In [7]:
# training loop

# a good range for learning rate with SGD
for lr in [0.003, 0.01, 0.03, 0.1]:
    # best: Learning rate: 0.03; test accuracy: 0.986

    # Each learning rate trial takes ~2 min at n_batches = 1000
    # so let's show that something is happening while we wait
    pbar = tqdm.tqdm(total=n_batches)

    torch.manual_seed(1)
    model = ConvModel()
    
    train_loader, test_loader = build_dataloaders(seed=1)

    # For reproducibility (see the GradSign section above for why we need this)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            
            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            loss.backward()
    
            with torch.no_grad():
                for p in model.parameters():
                    p -= lr * p.grad
            model.zero_grad()
            
            i += 1
            pbar.update(1)
            if i == n_batches:
                break

    pbar.close()
    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:40<00:00,  9.97it/s]


Learning rate: 0.003; test accuracy: 0.9722


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:35<00:00, 10.51it/s]


Learning rate: 0.01; test accuracy: 0.9836


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:40<00:00,  9.92it/s]


Learning rate: 0.03; test accuracy: 0.986


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:34<00:00, 10.59it/s]


Learning rate: 0.1; test accuracy: 0.9717


# Adam

In [8]:
#training loop

# a good range for learning rate with GradSign or Adam
for lr in [0.0001, 0.0003, 0.001, 0.003]:
    # Best: Learning rate: 0.0003; test accuracy: 0.982
    
    # Each learning rate trial takes ~2 min at n_batches = 1000
    # so let's show that something is happening while we wait
    pbar = tqdm.tqdm(total=n_batches)
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = Adam(model.parameters(), lr=lr)
    model.train()
    
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            pbar.update(1)
            if i == n_batches:
                break

    pbar.close()
    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:38<00:00, 10.19it/s]


Learning rate: 0.0001; test accuracy: 0.9794


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:37<00:00, 10.31it/s]


Learning rate: 0.0003; test accuracy: 0.9802


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:35<00:00, 10.47it/s]


Learning rate: 0.001; test accuracy: 0.982


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:40<00:00,  9.98it/s]


Learning rate: 0.003; test accuracy: 0.9712
